# 📊 W09 — Scenario Generation & Cost-Benefit Analysis

**Objective**: Demonstrate the decision support pipeline on a single unit.

**Pipeline**: RUL prediction → BI context → Scenario generation → Cost-benefit → Recommendation

**Author**: Fatima Khadija Benzine — March 2026

---
## 0. Setup

In [ ]:
import os
if not os.path.exists('/content/PhD-Project-'):
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
!pip install xgboost -q
os.chdir('/content/PhD-Project-/src')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json

project_root = Path('/content/PhD-Project-')
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = f'/content/drive/MyDrive/PhD_results/W09_{TIMESTAMP}'
os.makedirs(SAVE_DIR, exist_ok=True)

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer, create_sliding_windows, evaluate_per_unit
from bi_fusion import BIFusionPipeline, CONTINUOUS_BI_VARS
from feature_selection import BIAwareFeatureSelector
from ml_branch import MLBranch
from scenario_generator import (
    UnitContext, ScenarioGenerator, cost_benefit_table,
    generate_recommendation, plot_scenario_comparison,
    plot_degradation_trajectories,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Save: {SAVE_DIR}")
print("All imports ✓")

---
## 1. Load Data + Best ML Model

In [ ]:
DATASET = 'FD001'
W = 30
PAD = False

loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)
meta_cols = ['unit', 'cycle', 'rul']
train_raw = ds['train'].copy()
test_raw = ds['test'].copy()
train_raw['rul'] = train_raw['rul'].clip(upper=125)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=125)

sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]

norm = DataNormalizer(method='minmax')
train_norm = norm.fit_transform(train_raw, sensor_cols + setting_cols)
test_norm = norm.transform(test_raw)

fusion = BIFusionPipeline()
train_fused = fusion.fuse(train_norm, DATASET, split='train', encode_categoricals=True)
test_fused = fusion.fuse(test_norm, DATASET, split='test', encode_categoricals=True)
bi_cols = fusion.get_bi_columns(train_fused)
bi_cont = [c for c in CONTINUOUS_BI_VARS if c in train_fused.columns]
bi_norm = DataNormalizer(method='minmax')
train_fused = bi_norm.fit_transform(train_fused, bi_cont)
test_fused = bi_norm.transform(test_fused)

sel_corr = BIAwareFeatureSelector(variance_threshold=0.01, correlation_threshold=0.95)
fn_corr = sel_corr.select_features(data=train_fused, sensor_cols=sensor_cols,
    bi_cols=bi_cols, setting_cols=setting_cols, exclude_cols=meta_cols)
tr_corr = sel_corr.transform(train_fused, keep_cols=meta_cols)
te_corr = sel_corr.transform(test_fused, keep_cols=meta_cols)
X_train, y_train = create_sliding_windows(tr_corr, W, fn_corr, 'rul', pad=PAD)
X_test, y_test = create_sliding_windows(te_corr, W, fn_corr, 'rul', pad=PAD)

print(f"Data ready: {X_train.shape} train, {X_test.shape} test")

In [ ]:
# Load best ML params and train
RESULTS_W05 = '/content/drive/MyDrive/PhD_results/W05_20260303_0304'
with open(f'{RESULTS_W05}/A2_ml_grid_best_20260303_0304.json', 'r') as f:
    ml_saved = json.load(f)
best_ml = ml_saved['params']
print(f"ML params: {best_ml}")

ml_model = MLBranch(model_type='xgboost',
    flatten_strategy=best_ml['flatten_strategy'],
    n_estimators=best_ml['n_estimators'],
    max_depth=best_ml['max_depth'],
    learning_rate=best_ml['learning_rate_xgb'])
ml_model.fit(X_train, y_train, feature_names=fn_corr)

y_pred = ml_model.predict(X_test)
print(f"ML trained ✓ — predictions ready")

---
## 2. Select a Unit for Demonstration

In [ ]:
# Find last prediction per unit
unit_results = []
idx = 0
for u in sorted(te_corr['unit'].unique()):
    T = len(te_corr[te_corr['unit'] == u])
    n_win = max(T - (W - 1), 0)
    if n_win > 0:
        last_pred = y_pred[idx + n_win - 1]
        last_true = y_test[idx + n_win - 1]
        last_cycle = te_corr[te_corr['unit'] == u]['cycle'].max()
        
        # Get BI values for this unit (last row)
        unit_row = te_corr[te_corr['unit'] == u].iloc[-1]
        
        unit_results.append({
            'unit': u, 'cycle': last_cycle,
            'pred_rul': last_pred, 'true_rul': last_true,
            'start_idx': idx, 'n_windows': n_win,
            'downtime_penalty': unit_row.get('downtime_penalty', 0),
            'pm_cost': unit_row.get('pm_cost', 0),
            'cm_cost': unit_row.get('cm_cost', 0),
            'revenue_per_hour': unit_row.get('revenue_per_hour', 0),
            'technician_available': unit_row.get('technician_available', 1),
            'spare_parts_available': unit_row.get('spare_parts_available', 1),
            'spare_parts_lead_time': unit_row.get('spare_parts_lead_time', 0),
            'labor_rate_standard': unit_row.get('labor_rate_standard', 0),
            'labor_rate_overtime': unit_row.get('labor_rate_overtime', 0),
            'contract_penalty_active': unit_row.get('contract_penalty_active', 0),
        })
    idx += n_win

df_units = pd.DataFrame(unit_results)
print("=== Unit Overview (sorted by predicted RUL) ===")
print(df_units[['unit', 'cycle', 'pred_rul', 'true_rul', 'downtime_penalty']]
      .sort_values('pred_rul').head(10).to_string(index=False))

print(f"\n--- Critical units (pred RUL < 20) ---")
critical = df_units[df_units['pred_rul'] < 20]
print(critical[['unit', 'cycle', 'pred_rul', 'true_rul']].to_string(index=False))

In [ ]:
# Select a critical unit for detailed analysis
# Pick one with low predicted RUL and high downtime_penalty (interesting case)
UNIT_ID = critical.sort_values('downtime_penalty', ascending=False).iloc[0]['unit']
unit_info = df_units[df_units['unit'] == UNIT_ID].iloc[0]

print(f"\n=== Selected Unit {int(UNIT_ID)} ===")
print(f"  Current cycle:      {int(unit_info['cycle'])}")
print(f"  Predicted RUL:      {unit_info['pred_rul']:.1f} cycles")
print(f"  True RUL:           {unit_info['true_rul']:.0f} cycles")
print(f"  Downtime penalty:   {unit_info['downtime_penalty']:.3f}")
print(f"  PM cost:            {unit_info['pm_cost']:.3f}")
print(f"  CM cost:            {unit_info['cm_cost']:.3f}")
print(f"  CM/PM ratio:        {unit_info['cm_cost']/max(unit_info['pm_cost'], 0.001):.1f}x")

---
## 3. Build Unit Context from BI Data

In [ ]:
# Scale normalized BI values to realistic industrial costs
# These scaling factors simulate real-world magnitudes
COST_SCALE = {
    'pm_cost': 15000,            # $15,000 for preventive maintenance
    'cm_cost': 60000,            # $60,000 for corrective (emergency)
    'downtime_penalty': 2000,    # $2,000 per hour of downtime
    'revenue_per_hour': 5000,    # $5,000 revenue per operating hour
    'labor_rate_standard': 80,   # $80/hour standard rate
    'labor_rate_overtime': 120,  # $120/hour overtime
    'spare_parts_lead_time': 48, # 48 hours lead time
}

context = UnitContext(
    unit_id=int(UNIT_ID),
    current_cycle=int(unit_info['cycle']),
    predicted_rul=float(unit_info['pred_rul']),
    pm_cost=float(unit_info['pm_cost']) * COST_SCALE['pm_cost'],
    cm_cost=float(unit_info['cm_cost']) * COST_SCALE['cm_cost'],
    downtime_penalty=float(unit_info['downtime_penalty']) * COST_SCALE['downtime_penalty'],
    revenue_per_hour=float(unit_info['revenue_per_hour']) * COST_SCALE['revenue_per_hour'],
    technician_available=int(unit_info['technician_available']),
    spare_parts_available=int(unit_info['spare_parts_available']),
    spare_parts_lead_time=float(unit_info['spare_parts_lead_time']) * COST_SCALE['spare_parts_lead_time'],
    labor_rate_standard=float(unit_info['labor_rate_standard']) * COST_SCALE['labor_rate_standard'],
    labor_rate_overtime=float(unit_info['labor_rate_overtime']) * COST_SCALE['labor_rate_overtime'],
    contract_penalty_active=int(unit_info['contract_penalty_active']),
)

print(f"=== Unit Context ===")
print(f"  Unit {context.unit_id} at cycle {context.current_cycle}")
print(f"  Predicted RUL: {context.predicted_rul:.0f} cycles")
print(f"  Estimated EoL: cycle {context.eol}")
print(f"  PM cost:       ${context.pm_cost:,.0f}")
print(f"  CM cost:       ${context.cm_cost:,.0f} ({context.cm_to_pm_ratio:.1f}x PM)")
print(f"  Downtime:      ${context.downtime_penalty:,.0f}/hour")
print(f"  Revenue:       ${context.revenue_per_hour:,.0f}/hour")
print(f"  Technician:    {'Available' if context.technician_available else 'Not available'}")
print(f"  Spare parts:   {'Available' if context.spare_parts_available else 'Not available'}")

---
## 4. Generate Maintenance Scenarios

In [ ]:
generator = ScenarioGenerator(
    w_cost=0.6,
    w_risk=0.4,
    pm_downtime_hours=8,
    cm_downtime_hours=48,
)

scenarios = generator.generate_scenarios(
    context,
    n_pm_times=5,
    restoration_levels=[0.5, 0.7, 0.9],
)

print(f"Generated {len(scenarios)} scenarios")
print(f"  - 1 Do Nothing (baseline)")
print(f"  - {len(scenarios)-1} PM scenarios (5 times × 3 restoration levels)")

---
## 5. Cost-Benefit Analysis

In [ ]:
# Full comparison table
df_scenarios = cost_benefit_table(scenarios)

print(f"=== Scenario Ranking (best first) ===")
print(df_scenarios[['Scenario', 'Total Cost', 'Failure Risk', 'RUL After PM', 'Score']]
      .to_string(index=False))

In [ ]:
# Top 5 comparison
print("\n=== Top 5 Scenarios ===")
print(df_scenarios.head().to_string(index=False))

In [ ]:
# Cost breakdown table (for article)
top5 = df_scenarios.head(5)
print("\n=== Cost Breakdown (Top 5) ===")
print(top5[['Scenario', 'Intervention Cost', 'Downtime Cost',
            'Production Loss', 'Total Cost', 'Failure Risk', 'Score']]
      .to_string(index=False))

---
## 6. Recommendation

In [ ]:
rec = generate_recommendation(scenarios, context)
print(rec)

---
## 7. Visualizations

In [ ]:
plot_scenario_comparison(scenarios, context,
    save_path=f'{SAVE_DIR}/fig_scenario_comparison_{TIMESTAMP}.png')
plt.close()

In [ ]:
plot_degradation_trajectories(scenarios, context,
    save_path=f'{SAVE_DIR}/fig_degradation_trajectories_{TIMESTAMP}.png')
plt.close()

---
## 8. Save Results

In [ ]:
# Save scenario table
df_scenarios.to_csv(f'{SAVE_DIR}/scenario_table_{TIMESTAMP}.csv', index=False)

# Save recommendation
with open(f'{SAVE_DIR}/recommendation_{TIMESTAMP}.txt', 'w') as f:
    f.write(rec)

# Save context
with open(f'{SAVE_DIR}/unit_context_{TIMESTAMP}.json', 'w') as f:
    json.dump({
        'unit_id': context.unit_id,
        'current_cycle': context.current_cycle,
        'predicted_rul': context.predicted_rul,
        'pm_cost': context.pm_cost,
        'cm_cost': context.cm_cost,
        'downtime_penalty': context.downtime_penalty,
        'best_scenario': scenarios[0].name,
        'best_score': scenarios[0].score,
        'savings_vs_failure': scenarios[-1].total_cost - scenarios[0].total_cost
            if scenarios[-1].action == 'do_nothing'
            else scenarios[0].total_cost,
    }, f, indent=2)

print(f"\n✓ All saved to {SAVE_DIR}")

---
## Files saved:
```
PhD_results/W09_YYYYMMDD_HHMM/
    scenario_table_*.csv
    recommendation_*.txt
    unit_context_*.json
    fig_scenario_comparison_*.png
    fig_degradation_trajectories_*.png
```